# Feature Playground

Inspect the shared feature catalog, build shared features, and add notebook-local custom features on top.

## Running This Notebook In Colab

If you want to run this notebook in Google Colab, start by cloning the repo into the Colab session and installing the toolkit in editable mode.

Steps:

1. Set `REPO_URL` below to your GitHub repo URL.
2. Run the bootstrap cell once.
3. After that, the rest of the notebook can import `portfolio_toolkit` normally.

If you are running locally, the same cell will automatically fall back to the repository on your machine.


In [ ]:
# Colab / local bootstrap cell
# - In Colab: clone the repo, install the package, and point repo_root at /content/...
# - Locally: just point repo_root at this repository on disk

import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_URL = 'https://github.com/<your-user-or-org>/<your-repo>.git'  # replace with your real repo URL
    REPO_DIR = '/content/Portfolio-Optimizer'

    if '<your-user-or-org>' in REPO_URL or '<your-repo>' in REPO_URL:
        raise ValueError('Set REPO_URL to your real GitHub repository URL before running this cell in Colab.')

    !rm -rf "$REPO_DIR"
    !git clone "$REPO_URL" "$REPO_DIR"
    %cd "$REPO_DIR"
    %pip install -e ".[dev]"

    repo_root = Path(REPO_DIR).resolve()
else:
    repo_root = Path(repo_root).resolve() if 'repo_root' in globals() else Path('../../').resolve()

print('repo_root =', repo_root)


In [ ]:
from pathlib import Path
import pandas as pd

from portfolio_toolkit import build_features, list_features, load_prices, validate_feature_frame

repo_root = Path(repo_root).resolve() if 'repo_root' in globals() else Path('../../').resolve()
dataset_name = 'shared_set_2'
prices = load_prices(dataset_name, repo_root=repo_root)
available_features = list_features()
available_features[:20], len(available_features)

In [ ]:
shared = build_features(prices, feature_names=['return_5d', 'vol_20d', 'momentum_20d', 'price_to_sma_20d'])
shared['custom_return_minus_vol'] = shared['return_5d'] - shared['vol_20d'].fillna(0.0)
shared = validate_feature_frame(shared)
shared.head()

In [ ]:
# Validation cell
assert {'date', 'ticker', 'custom_return_minus_vol'}.issubset(shared.columns)
assert shared[['date', 'ticker']].duplicated().sum() == 0
print('Feature playground validated.')